In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").getOrCreate()

spark

25/09/16 19:15:28 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark

## Load One Month of NYC Taxi/Limousine Trip Data

In [3]:
%%sql

CREATE DATABASE IF NOT EXISTS nyc

25/09/16 20:16:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


++
||
++
++

In [4]:
%%sql

DROP TABLE IF EXISTS nyc.taxis

++
||
++
++

In [5]:
df = spark.read.parquet("/home/iceberg/data/yellow_tripdata_2025-01.parquet")
df.write.saveAsTable("nyc.taxis")

In [6]:
%%sql

DESCRIBE EXTENDED nyc.taxis

col_name,data_type,comment
VendorID,int,None
tpep_pickup_datetime,timestamp_ntz,None
tpep_dropoff_datetime,timestamp_ntz,None
passenger_count,bigint,None
trip_distance,double,None
RatecodeID,bigint,None
store_and_fwd_flag,string,None
PULocationID,int,None
DOLocationID,int,None
payment_type,bigint,None


In [7]:
%%sql

SELECT COUNT(*) as cnt
FROM nyc.taxis;

-- 3.4 M records

cnt
3475226


## Schema Evolution
Adding, dropping, renaming, or altering columns is easy and safe in Iceberg. In this example, we'll rename fare_amount to fare and trip_distance to distance. We'll also add a float column fare_per_distance_unit immediately after distance.

In [8]:
%%sql

ALTER TABLE nyc.taxis RENAME COLUMN fare_amount TO fare

++
||
++
++

In [9]:
%%sql

ALTER TABLE nyc.taxis RENAME COLUMN trip_distance TO distance

++
||
++
++

In [10]:
%%sql

ALTER TABLE nyc.taxis ALTER COLUMN distance COMMENT 'The elapsed trip distance in miles reported by the taximeter.'

++
||
++
++

In [11]:
%%sql

ALTER TABLE nyc.taxis ALTER COLUMN distance TYPE double;

++
||
++
++

In [12]:
%%sql

ALTER TABLE nyc.taxis ALTER COLUMN distance AFTER fare;

++
||
++
++

In [13]:
%%sql

ALTER TABLE nyc.taxis
ADD COLUMN fare_per_distance_unit float AFTER distance

++
||
++
++

Let's update the new fare_per_distance_unit to equal fare divided by distance.

In [14]:
%%sql

UPDATE nyc.taxis
SET fare_per_distance_unit = fare/distance

++
||
++
++

In [16]:
%%sql

DESCRIBE nyc.taxis

col_name,data_type,comment
VendorID,int,None
tpep_pickup_datetime,timestamp_ntz,None
tpep_dropoff_datetime,timestamp_ntz,None
passenger_count,bigint,None
RatecodeID,bigint,None
store_and_fwd_flag,string,None
PULocationID,int,None
DOLocationID,int,None
payment_type,bigint,None
fare,double,None


In [17]:
%%sql

SELECT
VendorID
,tpep_pickup_datetime
,tpep_dropoff_datetime
,fare
,distance
,fare_per_distance_unit
FROM nyc.taxis
LIMIT 10

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,fare,distance,fare_per_distance_unit
2,2025-01-17 18:08:04,2025-01-17 18:19:55,13.22,0.0,None
2,2025-01-17 18:16:20,2025-01-17 18:33:22,21.09,0.01,2109.0
1,2025-01-17 18:01:26,2025-01-17 18:22:49,25.7,6.7,3.8358209133148193
2,2025-01-17 18:51:18,2025-01-17 19:11:39,-4.75,0.0,None
2,2025-01-17 18:14:51,2025-01-17 18:25:51,10.04,0.0,None
2,2025-01-17 18:06:08,2025-01-17 18:11:01,8.11,0.0,None
1,2025-01-17 18:52:22,2025-01-17 18:58:24,6.03,0.8,7.537499904632568
2,2025-01-17 18:26:00,2025-01-17 18:54:38,32.62,0.0,None
2,2025-01-17 18:39:09,2025-01-17 18:39:25,-4.75,0.0,None
2,2025-01-17 18:47:11,2025-01-17 19:06:21,16.88,0.0,None


## Expressive SQL for Row Level Changes¶
With Iceberg tables, DELETE queries can be used to perform row-level deletes. 

If the filter matches an entire partition of the table, Iceberg will intelligently perform a metadata-only operation where it simply deletes the metadata for that partition.

Let's perform a row-level delete for all rows that have a fare_per_distance_unit greater than 4 or a distance greater than 2. 

In [18]:
%%sql

Select count(*) FROM nyc.taxis
WHERE fare_per_distance_unit > 4.0 OR distance > 2.0

count(1)
3316930


In [19]:
%%sql

DELETE FROM nyc.taxis
WHERE fare_per_distance_unit > 4.0 OR distance > 2.0;

++
||
++
++

There are some fares that have a null for fare_per_distance_unit due to the distance being 0. Let's remove those as well.

In [20]:
%%sql

Select count(*) FROM nyc.taxis
WHERE fare_per_distance_unit is null

count(1)
90893


In [21]:
%%sql

DELETE FROM nyc.taxis
WHERE fare_per_distance_unit is null

++
||
++
++

In [22]:
%%sql

SELECT COUNT(*) as cnt
FROM nyc.taxis

cnt
67403


## Partitioning
A table’s partitioning can be updated in place and applied only to newly written data. 

Query plans are then split, using the old partition scheme for data written before the partition scheme was changed, and using the new partition scheme for data written after. 
                                                                                                                                                       
People querying the table don’t even have to be aware of this split. 
                                                                                                                                                       
Simple predicates in WHERE clauses are automatically converted to partition filters that prune out files with no matches. 
                                                                                                                                                       
This is Hidden Partitioning in Iceberg.

In [23]:
%%sql

select * from demo.nyc.taxis.partitions	

record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
67403,2,1051384,0,0,0,0,2025-09-16 20:22:26.399000,3761776300079202307


In [27]:
%%sql

ALTER TABLE nyc.taxis
ADD PARTITION FIELD VendorID

IllegalArgumentException: Cannot add duplicate partition field null=ref(name="VendorID"), conflicts with 1000: VendorID: identity(1)

In [28]:
%%sql

select * from demo.nyc.taxis.partitions	

record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
67403,2,1051384,0,0,0,0,2025-09-16 20:22:26.399000,3761776300079202307


## Metadata Tables¶
Iceberg tables contain very rich metadata that can be easily queried. 

Retrieve the manifest list for any snapshot by querying the table's snapshots table.

In [29]:
%%sql

SELECT snapshot_id, manifest_list
FROM nyc.taxis.snapshots

snapshot_id,manifest_list
3288228855292588558,s3://warehouse/nyc/taxis/metadata/snap-3288228855292588558-1-57aae42a-8c64-42c2-8939-529bd36fd870.avro
1050162829464247387,s3://warehouse/nyc/taxis/metadata/snap-1050162829464247387-1-18cf9d0a-6778-4951-a9fa-83ba005a2c0c.avro
6938044215708356115,s3://warehouse/nyc/taxis/metadata/snap-6938044215708356115-1-7d6ee602-c9f8-4713-aa2c-70eef9c6c464.avro
3761776300079202307,s3://warehouse/nyc/taxis/metadata/snap-3761776300079202307-1-17628c98-5bd9-4a4d-ac6a-386183fb0675.avro


### The files table contains loads of information on data files, including column level statistics such as null counts, lower bounds, and upper bounds.

In [30]:
%%sql

SELECT file_path, file_format, record_count, null_value_counts, lower_bounds, upper_bounds
FROM nyc.taxis.files

file_path,file_format,record_count,null_value_counts,lower_bounds,upper_bounds
s3://warehouse/nyc/taxis/data/00000-45-effa374a-468d-4ff1-9a29-16f67f38022b-0-00001.parquet,PARQUET,35377,"{1: 0, 2: 0, 3: 0, 4: 14438, 5: 0, 6: 14438, 7: 14438, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0, 16: 0, 17: 0, 18: 14438, 19: 14438, 20: 0}","{1: bytearray(b'\x01\x00\x00\x00'), 2: bytearray(b'\x00j\xe0\xc7\x99*\x06\x00'), 3: bytearray(b'\x00M\x04\xe9\x99*\x06\x00'), 4: bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00'), 5: bytearray(b'{\x14\xaeG\xe1z\x84?'), 6: bytearray(b'\x01\x00\x00\x00\x00\x00\x00\x00'), 7: bytearray(b'N'), 8: bytearray(b'\x01\x00\x00\x00'), 9: bytearray(b'\x01\x00\x00\x00'), 10: bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00'), 11: bytearray(b'\x00\x00\x00\x00\x00\xe0\x85\xc0'), 12: bytearray(b'\x00\x00\x00\x00\x00\x00\x1e\xc0'), 13: bytearray(b'\x00\x00\x00\x00\x00\x00\xe0\xbf'), 14: bytearray(b'fffff\x06I\xc0'), 15: bytearray(b'=\n\xd7\xa3p}7\xc0'), 16: bytearray(b'\x00\x00\x00\x00\x00\x00\xf0\xbf'), 17: bytearray(b'\x00\x00\x00\x00\x00\x02\x86\xc0'), 18: bytearray(b'\x00\x00\x00\x00\x00\x00\x04\xc0'), 19: bytearray(b'\x00\x00\x00\x00\x00\x00\xfc\xbf'), 20: bytearray(b'\x00\xc8\x12\xc6')}","{1: bytearray(b'\x06\x00\x00\x00'), 2: bytearray(b'\xc0\xab\x97V\t-\x06\x00'), 3: bytearray(b'\x00Stt\t-\x06\x00'), 4: bytearray(b'\x06\x00\x00\x00\x00\x00\x00\x00'), 5: bytearray(b'\x00\x00\x00\x00\x00\x00\x00@'), 6: bytearray(b'c\x00\x00\x00\x00\x00\x00\x00'), 7: bytearray(b'Y'), 8: bytearray(b'\t\x01\x00\x00'), 9: bytearray(b'\t\x01\x00\x00'), 10: bytearray(b'\x04\x00\x00\x00\x00\x00\x00\x00'), 11: bytearray(b'\xcd\xcc\xcc\xcc\xcc\xcc\x1f@'), 12: bytearray(b'\x00\x00\x00\x00\x00\x00\x1b@'), 13: bytearray(b'\x00\x00\x00\x00\x00\x00\xe0?'), 14: bytearray(b'\x00\x00\x00\x00\x00@e@'), 15: bytearray(b'\x8f\xc2\xf5(\\\x0f@@'), 16: bytearray(b'\x00\x00\x00\x00\x00\x00\xf0?'), 17: bytearray(b'\x00\x00\x00\x00\x008f@'), 18: bytearray(b'\x00\x00\x00\x00\x00\x00\x04@'), 19: bytearray(b'\x00\x00\x00\x00\x00\x00\xfc?'), 20: bytearray(b'\x00\x00\x80@')}"
s3://warehouse/nyc/taxis/data/00001-46-effa374a-468d-4ff1-9a29-16f67f38022b-0-00001.parquet,PARQUET,32026,"{1: 0, 2: 0, 3: 0, 4: 19563, 5: 0, 6: 19563, 7: 19563, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0, 16: 0, 17: 0, 18: 19563, 19: 19563, 20: 0}","{1: bytearray(b'\x01\x00\x00\x00'), 2: bytearray(b'\xc0/w\xba\x99*\x06\x00'), 3: bytearray(b'\x80\xc1\xcd\xc6\x99*\x06\x00'), 4: bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00'), 5: bytearray(b'{\x14\xaeG\xe1z\x84?'), 6: bytearray(b'\x01\x00\x00\x00\x00\x00\x00\x00'), 7: bytearray(b'N'), 8: bytearray(b'\x01\x00\x00\x00'), 9: bytearray(b'\x03\x00\x00\x00'), 10: bytearray(b'\x00\x00\x00\x00\x00\x00\x00\x00'), 11: bytearray(b'\x00\x00\x00\x00\x00 \x8c\xc0'), 12: bytearray(b'\x00\x00\x00\x00\x00\x00\x1e\xc0'), 13: bytearray(b'\x00\x00\x00\x00\x00\x00\xe0\xbf'), 14: bytearray(b'\x00\x00\x00\x00\x00\x80U\xc0'), 15: bytearray(b'q=\n\xd7\xa3\xf0;\xc0'), 16: bytearray(b'\x00\x00\x00\x00\x00\x00\xf0\xbf'), 17: bytearray(b'\x00\x00\x00\x00\x00(\x8c\xc0'), 18: bytearray(b'\x00\x00\x00\x00\x00\x00\x04\xc0'), 19: bytearray(b'\x00\x00\x00\x00\x00\x00\xfc\xbf'), 20: bytearray(b'\x00p\x14\xc6')}","{1: bytearray(b'\x06\x00\x00\x00'), 2: bytearray(b'\x00\xdc,V\t-\x06\x00'), 3: bytearray(b'\x00\xa50\x84\t-\x06\x00'), 4: bytearray(b'\x06\x00\x00\x00\x00\x00\x00\x00'), 5: bytearray(b'\x00\x00\x00\x00\x00\x00\x00@'), 6: bytearray(b'c\x00\x00\x00\x00\x00\x00\x00'), 7: bytearray(b'Y'), 8: bytearray(b'\t\x01\x00\x00'), 9: bytearray(b'\t\x01\x00\x00'), 10: bytearray(b'\x04\x00\x00\x00\x00\x00\x00\x00'), 11: bytearray(b'\x8f\xc2\xf5(\\\x8f\x1f@'), 12: bytearray(b'\x00\x00\x00\x00\x00\x00\x17@'), 13: bytearray(b'\x00\x00\x00\x00\x00\x00\xe0?'), 14: bytearray(b'\xcd\xcc\xcc\xcc\xcc,T@'), 15: bytearray(b'R\xb8\x1e\x85\xebQ6@'), 16: bytearray(b'\x00\x00\x00\x00\x00\x00\xf0?'), 17: bytearray(b'\n\xd7\xa3p=\xaaO@'), 18: bytearray(b'\x00\x00\x00\x00\x00\x00\x04@'), 19:

## Time Travel¶
The history table lists all snapshots and which parent snapshot they derive from. 
The is_current_ancestor flag let's you know if a snapshot is part of the linear history of the current snapshot of the table.

In [31]:
%%sql --var df

SELECT *
FROM nyc.taxis.history

made_current_at,snapshot_id,parent_id,is_current_ancestor
2025-09-16 20:17:04.746000,3288228855292588558,None,True
2025-09-16 20:19:58.950000,1050162829464247387,3288228855292588558,True
2025-09-16 20:22:16.411000,6938044215708356115,1050162829464247387,True
2025-09-16 20:22:26.399000,3761776300079202307,6938044215708356115,True


### Time-travel by altering the current-snapshot-id property of the table to reference any snapshot in the table's history. 
Let's revert the table to it's original state by traveling to the very first snapshot ID.

In [32]:
original_snapshot = df.head().snapshot_id
spark.sql(f"CALL system.rollback_to_snapshot('nyc.taxis', {original_snapshot})")
original_snapshot

3288228855292588558

In [33]:
%%sql

SELECT
VendorID
,tpep_pickup_datetime
,tpep_dropoff_datetime
,fare
,distance
,fare_per_distance_unit
FROM nyc.taxis
LIMIT 10

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,fare,distance,fare_per_distance_unit
1,2025-01-01 00:18:38,2025-01-01 00:26:59,10.0,1.6,None
1,2025-01-01 00:32:40,2025-01-01 00:35:13,5.1,0.5,None
1,2025-01-01 00:44:04,2025-01-01 00:46:01,5.1,0.6,None
2,2025-01-01 00:14:27,2025-01-01 00:20:01,7.2,0.52,None
2,2025-01-01 00:21:34,2025-01-01 00:25:06,5.8,0.66,None
2,2025-01-01 00:48:24,2025-01-01 01:08:26,19.1,2.63,None
1,2025-01-01 00:14:47,2025-01-01 00:16:15,4.4,0.4,None
1,2025-01-01 00:39:27,2025-01-01 00:51:51,12.1,1.6,None
1,2025-01-01 00:53:43,2025-01-01 01:13:23,19.1,2.8,None
2,2025-01-01 00:00:02,2025-01-01 00:09:36,11.4,1.71,None


In [33]:
%%sql

SELECT *
FROM nyc.taxis.history

made_current_at,snapshot_id,parent_id,is_current_ancestor
2025-09-16 09:08:31.800000,5169512661232869107,None,True
2025-09-16 09:12:02.419000,5451652599387697692,5169512661232869107,False
2025-09-16 09:17:25.265000,3863296958651222306,5451652599387697692,False
2025-09-16 09:18:18.731000,5664066731713503762,3863296958651222306,False
2025-09-16 09:31:55.524000,5169512661232869107,None,True


In [34]:
%%sql

SELECT COUNT(*) as cnt
FROM nyc.taxis

-- 3.4 M records

cnt
3475226


In [35]:
%%sql

SHOW CURRENT NAMESPACE;

catalog,namespace
demo,
